# Understanding the Data

This notebook explains the two data components in the pipeline:
1. **Embeddings** - numerical representations of audio segments
2. **Labels** - the class for each segment

## What are embeddings?

Embeddings are high-dimensional vectors that represent audio in numerical form. Instead of working with raw waveforms, we use pre-trained models (like BirdNET) to convert each audio segment into a fixed-size vector (e.g. 1024 dimensions). These vectors capture the acoustic content of each segment in a form that a classifier can learn from.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

# Resolve project root (two levels up from core/docs/)
PROJECT_ROOT = Path().absolute().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

EMBEDDINGS_DIR = PROJECT_ROOT / "ESC10_BASEAL" / "embeddings" / "birdnet"
LABELS_PATH = PROJECT_ROOT / "ESC10_BASEAL" / "labels.csv"

# Load a single embedding file to see the shape
embedding_files = sorted(EMBEDDINGS_DIR.glob("*.npy"))
print(f"Found {len(embedding_files)} embedding files")

if embedding_files:
    example = np.load(embedding_files[0])
    print(f"\nExample file: {embedding_files[0].name}")
    print(f"Shape: {example.shape}")
    print(f"This is a single {example.shape[0]}-dimensional embedding vector")

Found 800 embedding files

Example file: 1-100032-A-0_000_003_birdnet.npy
Shape: (1024,)
This is a single 1024-dimensional embedding vector


## Understanding the labels

Labels are stored in a CSV file. Each row maps a filename to its class label.

In [2]:
df = pd.read_csv(LABELS_PATH)
print("Columns:", df.columns.tolist())
print(f"Total samples: {len(df)}")
print(f"\nFirst few rows:")
print(df.head(10))
print(f"\nUnique labels ({len(df['label'].unique())}):")
print(df['label'].unique())

Columns: ['filename', 'label', 'validation']
Total samples: 800

First few rows:
                    filename           label  validation
0   1-100032-A-0_000_003.mp3             dog       False
1   1-100032-A-0_003_006.mp3             dog       False
2   1-110389-A-0_000_003.mp3             dog       False
3   1-110389-A-0_003_006.mp3             dog       False
4  1-116765-A-41_000_003.mp3        chainsaw       False
5  1-116765-A-41_003_006.mp3        chainsaw       False
6   1-17150-A-12_000_003.mp3  crackling_fire       False
7   1-17150-A-12_003_006.mp3  crackling_fire       False
8  1-172649-A-40_000_003.mp3      helicopter       False
9  1-172649-A-40_003_006.mp3      helicopter       False

Unique labels (10):
['dog' 'chainsaw' 'crackling_fire' 'helicopter' 'rain' 'crying_baby'
 'clock_tick' 'sneezing' 'rooster' 'sea_waves']


## How embeddings and labels are matched

Each row in `labels.csv` has a `filename` field (e.g. `1-100032-A-0_000_003.mp3`). The corresponding embedding file is found by taking the filename stem and appending the model name:

```
1-100032-A-0_000_003.mp3  -->  1-100032-A-0_000_003_birdnet.npy
```

The `ActiveLearner._load_data()` method handles this matching automatically. It loads all matching pairs and creates two parallel arrays:
- `embeddings` of shape (N, 1024)
- `labels` of shape (N,)

where N is the number of successfully matched samples.

In [3]:
# Demonstrate the matching
matched = 0
missing = 0

for _, row in df.head(20).iterrows():
    stem = Path(row['filename']).stem
    embedding_path = EMBEDDINGS_DIR / f"{stem}_birdnet.npy"
    if embedding_path.exists():
        matched += 1
    else:
        missing += 1
        print(f"  Missing: {embedding_path.name}")

print(f"\nMatched: {matched}/20, Missing: {missing}/20")


Matched: 20/20, Missing: 0/20


## The flattened dataset

After matching, the pipeline creates a flat dataset where each sample is just an index (0, 1, 2, ... N-1). The original filenames are no longer needed during training. Active learning works with these indices to track which samples are labelled and which are unlabelled.

```
Embedding files + labels CSV
    |
    v
Matching (filename stem --> embedding file)
    |
    v
Flat arrays: embeddings (N, 1024) + labels (N,)
    |
    v
Active learning uses indices [0, 1, 2, ..., N-1]
```

## Using your own data

To use your own dataset, create a folder with this structure:

```
YOUR_DATASET/
    embeddings/
        birdnet/              # or whichever model you used
            file1_birdnet.npy
            file2_birdnet.npy
            ...
    labels.csv                # columns: filename, label
```

Then update the paths in `core/config.yml` to point to your dataset.